# **Pangenome Analysis – Panaroo**

## **Tool Information**

- **Tool:** Panaroo (v1.5.2)  
- **Input:** Prokka-annotated GFF3 files  
- **Organism:** *Acinetobacter baumannii*  
- **Analysis type:** Pangenome construction and core genome alignment  

Panaroo is a graph-based pangenome analysis tool designed to construct high-quality pangenomes while correcting common annotation errors such as fragmented genes, misannotations, and contamination.

It improves upon traditional pangenome tools by using a graph-based clustering approach, reducing inflated accessory genome estimates and producing more biologically accurate gene clusters.

This notebook documents the construction of the *Acinetobacter baumannii* pangenome and generation of a core genome alignment for downstream phylogenetic and comparative genomic analyses.

## **Create a Dedicated Environment**

We create a dedicated conda environment to isolate Panaroo and its dependencies from other tools. This ensures a clean and reproducible setup for pangenome analysis workflows.

In [ ]:
%%bash

conda create -n panaroo_aba python=3.9 -y

## **Install Panaroo from GitHub**

We install Panaroo directly from the official GitHub repository to ensure access to the latest compatible version. This approach helps avoid package availability and dependency issues commonly encountered with standard pip installations.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate panaroo_aba

pip install git+https://github.com/gtonkinhill/panaroo.git

## **Install Panaroo Dependencies**

We install required dependencies such as CD-HIT and MAFFT to support clustering and alignment steps. These tools are essential for accurate pangenome construction and downstream analyses.

In [ ]:
%%bash

# Initialize conda
source /home/anaconda/miniconda3/etc/profile.d/conda.sh

# Activate environment
conda activate panaroo_aba

# Install dependencies
conda install -c bioconda cd-hit mafft -y

## **Verify Installation**

We verify that Panaroo is installed correctly by checking the tool version. This confirms that the installation was successful and the tool is ready for pangenome analysis.

In [ ]:
%%bash
# Load conda into this Jupyter shell session
source /home/anaconda/miniconda3/etc/profile.d/conda.sh

# Activate Panaroo environment
conda activate panaroo_aba

echo "[INFO] Panaroo version:"
panaroo --version

[INFO] Panaroo version:
panaroo 1.5.2


## **Input Requirements**

Panaroo needs a `text file` containing paths to all `.gff` annotation files, typically generated using Prokka.

Each gff file should contain:

1. Gene annotations (CDS, rRNA, tRNA)  
2. Consistent locus tags  
3. Standardized formatting (Prokka-compatible)  

⚠️ Ensure that:

- All genomes are annotated using the same pipeline (e.g., Prokka)  
- File naming is consistent  
- No duplicate or corrupted files are present

The following Bash cell collects all valid GFF files produced by Prokka into a text file.

In [2]:
%%bash

PROKKA_DIR=/data/internship_data/nidhi/aba/new_output/prokka_output
PANAROO_DIR=/data/internship_data/nidhi/aba/new_output/panaroo_output

mkdir -p $PANAROO_DIR

# Create input list file
INPUT_LIST=$PANAROO_DIR/gff_list.txt
> $INPUT_LIST

# Collect all GFF files
for d in $PROKKA_DIR/*; do
    sample=$(basename "$d")
    gff="$d/$sample.gff"
    if [ -s "$gff" ]; then
        echo "$gff" >> $INPUT_LIST
    else
        echo "Skipping $sample — missing or empty GFF"
    fi
done

echo "GFF list saved to: $INPUT_LIST"
echo "Number of GFFs:"
wc -l $INPUT_LIST

Skipping master_prokka.log — missing or empty GFF
GFF list saved to: /data/internship_data/nidhi/aba/new_output/panaroo_output/gff_list.txt
Number of GFFs:
71 /data/internship_data/nidhi/aba/new_output/panaroo_output/gff_list.txt


## **Execution Strategy**

Panaroo was run in **strict clean-mode** to minimize annotation errors and gene fragmentation, which is especially important for large datasets with potential assembly inconsistencies. A core gene threshold of 95% was used to define genes present in the majority of isolates.

**NOTE:** This step may take 2–4 hours depending on CPU + I/O speed.

The workflow includes:

- Clustering of orthologous genes across genomes  
- Correction of annotation errors  
- Construction of gene presence/absence matrix  
- Generation of core genome alignment  

A stringent mode was applied to improve clustering accuracy and reduce false gene splits.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate panaroo

PANAROO_DIR=/data/internship_data/nidhi/aba/new_output/panaroo_output
INPUT_LIST=$PANAROO_DIR/gff_list.txt

echo "[INFO] Starting Panaroo..."

panaroo \
    -a core \
    --clean-mode strict \
    -i $INPUT_LIST \
    -o $PANAROO_DIR \
    --threads 12 \
    --aligner mafft \
    --remove-invalid-genes \
    --core_threshold 0.95

echo "[INFO] Panaroo run completed."

## **Pangenome Workflow**

### Graph-based Gene Clustering

Panaroo constructs a graph where:

- Nodes represent gene clusters  
- Edges represent adjacency relationships across genomes  

This approach allows:

- Correction of fragmented genes  
- Merging of incorrectly split gene clusters  
- Removal of spurious annotations


## **Expected Output Files**

After completion, Panaroo generates multiple files including:

| Output File | Description |
|---|---|
| `core_alignment.aln` | Multiple sequence alignment of core genes used for phylogenetic analysis |
| `gene_presence_absence.csv` | Matrix showing gene distribution across isolates for pangenome analysis |
| `filtered_polymorphic_sites.fasta` | SNP-only alignment derived from the core genome |
| `summary_statistics.txt` | Summary statistics including core genome size, accessory genome size, and gene clusters |
| `gene_data.csv` | Metadata and clustering information for identified genes |
| `core_gene_alignment.aln` | Core genome alignment for downstream phylogenetic reconstruction |
| `pan_genome_reference.fa` | Representative reference sequences for the pangenome |
| `struct_presence_absence.Rtab` | Binary gene presence–absence matrix across isolates |
| `*.gml` | Graph representation of the pangenome network |
| `*.csv` | Additional gene clustering and annotation summary files |

In [13]:
%%bash

# List Panaroo output files
ls -lh /data/internship_data/nidhi/aba/new_output/panaroo_output

total 1.2G
drwxrwsr-x+ 2 nidhi nidhi 120K Apr  7 18:21 aligned_gene_sequences
-rw-rw-r--  1 nidhi nidhi  98K Apr  7 18:22 alignment_entropy.csv
-rw-rw-r--  1 nidhi nidhi 246M Apr  7 15:45 combined_DNA_CDS.fasta
-rw-rw-r--  1 nidhi nidhi  84M Apr  7 15:45 combined_protein_CDS.fasta
-rw-rw-r--  1 nidhi nidhi 2.6M Apr  7 15:43 combined_protein_cdhit_out.txt
-rw-rw-r--  1 nidhi nidhi 8.5M Apr  7 15:43 combined_protein_cdhit_out.txt.clstr
-rw-rw-r--  1 nidhi nidhi 330K Apr  7 18:22 core_alignment_filtered_header.embl
-rw-rw-r--  1 nidhi nidhi 364K Apr  7 18:22 core_alignment_header.embl
-rw-rw-r--  1 nidhi nidhi 203M Apr  7 18:22 core_gene_alignment.aln
-rw-rw-r--  1 nidhi nidhi 179M Apr  7 18:22 core_gene_alignment_filtered.aln
-rw-rw-r--  1 nidhi nidhi  34M Apr  7 15:45 final_graph.gml
-rw-rw-r--  1 nidhi nidhi 339M Apr  7 15:45 gene_data.csv
-rw-rw-r--  1 nidhi nidhi 873K Apr  7 15:45 gene_presence_absence.Rtab
-rw-rw-r--  1 nidhi nidhi 4.3M Apr  7 15:45 gene_presence_absence.csv
-rw-rw-

## **Citation**

Tonkin-Hill G, MacAlasdair N, Ruis C,
Weimann A, Horesh G, Lees JA, et al.

Producing polished prokaryotic pangenomes
with the Panaroo pipeline.

Genome Biology.
2020;21:180.

https://doi.org/10.1186/s13059-020-02090-4